# PCG Convergence Study  
Using NGSolve's built-in pcg solver and our finite element space hierarchy  


In [2]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
import time
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src import multigrid_cycles
from multigrid_cycles import *
from src import NGSolve_utils
from NGSolve_utils import *

In [3]:
# Getting colors for plots
cmap = plt.get_cmap('twilight_shifted')
cmap_nums = [tuple(cmap(x)[:3]) for x in np.linspace(0, 1,)]
# Setting options for plots
draw_opts = dict(
    Fullscreen=True,
    deformation=True,
    colors=cmap_nums,
    radius=0.75,
    center=[0.5, 0.5, 0.5],
    settings={
        "Objects": {"Wireframe":False},
        "camera": {
            "transformations":[
                {"type": "rotateX", "angle": -45}
            ]
        },
    },
)


In [4]:
# Setting up the problem we will solve on every level

# Boundary conditions
DIRICHLET = "left|right"

# LHS bilinear form
def poisson_bilinear(a, u, v):
    a += InnerProduct(grad(u), grad(v)) * dx

# RHS linear form
rhs_cf = 0
def poisson_linear(f, u, v):
    f += rhs_cf * v * dx

# Initial iterate
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))


In [ ]:

# Call our setup function to put these together
poisson_setup = build_form_setup(bilinear=poisson_bilinear, linear=poisson_linear)

# Define the coarsest mesh for our hierarchy of many sized meshes
N = 16
coarsest_mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
parfait = build_hierarchy(
    coarsest_mesh,
    poisson_setup,
    n_refines=5,
    order=1,
    dirichlet=DIRICHLET,
    dirichlet_value={"left": 0.0, "right":0.0},
    verbose=True,
)


  lev     ndof              A       P(c->f)    nfree  nfixed
  ---  -------  -------------  ------------  -------  ------
    0      339        339x339             -      305      34  (coarse)
    1     1289      1289x1289      1289x339     1223      66
    2     5025      5025x5025     5025x1289     4895     130
    3    19841    19841x19841    19841x5025    19583     258
    4    78849    78849x78849   78849x19841    78335     514
    5   314369  314369x314369  314369x78849   313343    1026
    6  1255425  1255425x1255425  1255425x314369  1253375    2050  (fine)


In [ ]:
finest = parfait.finest
finest.set_initial_guess(x0)

initial_plot = Draw(
        finest.gfu,
        finest.mesh,
        "initial guess (finest)",
        **draw_opts,
        )

In [ ]:
x = finest.gfu.vec
b = finest.f.vec
finest.enforce_dirichlet(x)
print("residual norm before solve",finest.residual_norm(b,x))
finest.coarse_solve(b,x)
print("residual norm after direct solve",finest.residual_norm())

In [ ]:
finest.set_initial_guess(x0)
finest.refresh()
restart_plot = Draw(
        finest.gfu,
        finest.mesh,
        "going back to initial guess (finest)",
        **draw_opts,
        )

In [ ]:
c = Preconditioner(finest.a, "local")
c.Update()
x2 = finest.gfu.vec
print("residual norm before CG", finest.residual_norm(b,x2))
solvers.BVP(bf=finest.a, lf=finest.f, gf=finest.gfu, pre=c, maxsteps=500, tol=1e-10, print=False)
cg_sol = finest.gfu.vec
print("residual norm after CG", finest.residual_norm(b,cg_sol))
cg_plot = Draw(finest.gfu, finest.mesh, "solution from CG", **draw_opts)

In [ ]:
level_norms = []
for lvl in reversed(parfait.levels):
    level = lvl
    print("This level has ", level.ndof, " degrees of freedom")
    level.set_initial_guess(x0)
    blevel = level.f.vec
    initial_plot = Draw(
        level.gfu,
        level.mesh,
        **draw_opts,
        )
    clevel = Preconditioner(level.a, "local")
    clevel.Update()
    xl = level.gfu.vec
    print("Before solve, the residual norm is ", level.residual_norm(blevel,xl))
    solvers.BVP(bf=level.a, lf=level.f, gf=level.gfu, pre=clevel, maxsteps=500, tol=1e-10, print=True)
    cg_sol = level.gfu.vec
    level_norms.append(level.residual_norm(blevel, cg_sol))
    print("After CG, the residual norm is ", level_norms[-1])
    solved_plot = Draw(level.gfu, level.mesh, **draw_opts)

print("Here are the errors after CG for all the levels:")
for l in level_norms: print(l, '\n')

